# LoRA Fine-tuning 공통 스켈레톤

Language·Vision·Speech 중 **하나**를 선택합니다. TODO를 완성하기 전에는 전체 실행이 되지 않습니다. 태스크별 모델·데이터·전처리·학습 설정은 직접 구현합니다.

Colab GPU 런타임에서 작업합니다. 동일한 Test 데이터로 학습 전후를 비교하고, 최소 5개 예측을 기록합니다.

## 1. 과제 설정

모델·데이터셋 이름과 선택 이유를 작성합니다. 예시는 영화 리뷰 감성 분석, 이미지 분류, 음성 명령 분류입니다.

In [1]:
TASK = "Language"  # Language / Vision / Speech
MODEL_ID = "klue/bert-base"
DATASET_ID = "klue/klue"
SEED = 42
OUTPUT_DIR = "./assignment_runs/news"


선택 이유: 뉴스 제목을 보고 경제 / 정치 / 스포츠 등 7개 주제로 분류하는 모델로,모델의 학습 과정과 파인튜닝 과정을 이해하기에 적합하다 생각하였음

## 2. 실행 환경

아래 설치 코드는 공통 라이브러리를 설치하는 기본 예시입니다. 먼저 설치 셀과 import 셀을 실행하여 정상 작동하는지 확인합니다. 버전 충돌이 발생하면 오류 메시지와 공식 문서를 참고하여 설치할 버전을 조정합니다.


In [2]:
%pip install transformers datasets peft accelerate pandas
%pip uninstall -y torchao


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
import importlib.metadata
import torch
import pandas as pd
from datasets import load_dataset
from transformers import pipeline, set_seed
from peft import LoraConfig, get_peft_model

set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
for package in ("torch", "transformers", "datasets", "peft", "accelerate"):
    print(package, importlib.metadata.version(package))


GPU: Tesla T4
torch 2.14.1+cu130
transformers 5.22.1
datasets 4.8.5
peft 0.21.2
accelerate 1.15.0


## 3. 데이터 로딩과 분할

load_dataset으로 데이터를 불러와 Dataset 또는 DatasetDict를 반환합니다. 분할 이름이 다르면 train / validation / test로 맞춥니다. 제공된 함수가 기존 분할을 유지하고 없는 분할만 생성합니다. 분할이 없으면 80% / 10% / 10%로 나눕니다.

분할 결과의 크기와 클래스 분포를 확인합니다. 같은 화자·원본을 묶어야 하는 데이터에는 무작위 분할 대신 그룹별 분할을 적용합니다.


In [4]:
from datasets import Dataset, DatasetDict

def prepare_splits(data, seed=SEED):
    splits = DatasetDict({"train": data}) if isinstance(data, Dataset) else DatasetDict(data)
    if "train" not in splits:
        raise ValueError("학습용 분할의 이름을 train으로 지정하세요.")
    if "test" not in splits:
        split = splits["train"].train_test_split(test_size=0.1, seed=seed)
        splits["train"], splits["test"] = split["train"], split["test"]
        validation_ratio = 1 / 9
    else:
        validation_ratio = 0.1
    if "validation" not in splits:
        split = splits["train"].train_test_split(test_size=validation_ratio, seed=seed)
        splits["train"], splits["validation"] = split["train"], split["test"]
    return DatasetDict({name: splits[name] for name in ("train", "validation", "test")})

def load_data():
  return load_dataset(DATASET_ID, "ynat")

raw = prepare_splits(load_data())
print(raw)
print(raw["train"][0])


DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 41110
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
    test: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 4568
    })
})
{'guid': 'ynat-v1_train_22408', 'title': '그래픽 숫자로 본 박근혜 재판 기록들종합', 'label': 6, 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=100&sid2=264&oid=001&aid=0010008080', 'date': '2018.04.06. 오전 10:44'}


In [5]:
label_names = raw["train"].features["label"].names
print("주제 목록:", label_names)

preview = raw["train"].select(range(5)).to_pandas()
preview["topic"] = preview["label"].map(lambda x: label_names[x])

display(preview[["title", "label", "topic"]])

# 각 분할에 주제별 데이터가 얼마나 있는지 확인
counts = pd.DataFrame({
    split: pd.Series(raw[split]["label"]).value_counts()
    for split in ("train", "validation", "test")
}).fillna(0).astype(int)

counts.index = [label_names[int(i)] for i in counts.index]
display(counts)

주제 목록: ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']


,title,label,topic
0,그래픽 숫자로 본 박근혜 재판 기록들종합,6,정치
1,투스크 EU 회원국에 노트르담성당 재건사업 동참 요청,4,세계
2,신간 모던 슈트 스토리·그림 창문 거울,3,생활문화
3,신한카드 상가 관리비 카드 자동이체 서비스,1,경제
4,민주 12일 첫 대규모 장외투쟁…거리로 나서는 제1야당,6,정치


,train,validation,test
IT과학,4714,554,521
경제,5499,1348,619
사회,4624,3701,509
생활문화,5203,1369,548
세계,7485,835,835
스포츠,6953,578,789
정치,6632,722,747


## 4. 모델과 전처리

Tokenizer / Image processor / Audio feature extractor 등을 선택합니다. 새 Classification head를 쓰면 초기화 사실과 레이블 매핑을 기록합니다. map으로 모델 입력과 labels를 구성합니다.

- Language: 텍스트·레이블 처리. 생성형 방식은 Prompt와 정답 구간의 Loss 처리를 결정합니다.
- Vision: 이미지 크기·정규화 처리
- Speech: Sampling rate·길이 처리

추론에서도 같은 전처리 의미를 유지합니다. Test 데이터에 학습용 무작위 증강을 적용하지 않습니다.

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MAX_LENGTH = 128

label_names = raw["train"].features["label"].names
id2label = dict(enumerate(label_names))
label2id = {name: idx for idx, name in id2label.items()}


def load_model_and_processor():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id,
    )
    return model, tokenizer


model, processor = load_model_and_processor()


def preprocess(example):
    inputs = processor(
        example["title"],
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs["labels"] = example["label"]
    return inputs


prepared = raw.map(
    preprocess,
    batched=True,
    remove_columns=raw["train"].column_names,
)

sample = prepared["train"][0]

assert len(sample["input_ids"]) <= MAX_LENGTH
assert 0 <= sample["labels"] < len(label_names)

print("전처리된 열:", prepared["train"].column_names)
print("첫 제목:", raw["train"][0]["title"])
print("토큰 번호 일부:", sample["input_ids"][:20])
print("토큰 개수:", len(sample["input_ids"]))
print("정답:", sample["labels"], id2label[sample["labels"]])
print("분류 주제 수:", model.config.num_labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


전처리된 열: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
첫 제목: 그래픽 숫자로 본 박근혜 재판 기록들종합
토큰 번호 일부: [2, 11531, 6197, 2200, 1163, 4655, 4503, 3809, 2031, 27854, 3]
토큰 개수: 11
정답: 6 정치
분류 주제 수: 7


전처리·레이블 매핑·Classification head 초기화 여부: 전처리 과정에서는 토크나이저로 뉴스 제목을 최대 128토큰으로 변환하였고, 주제 번호 0~6을 정답으로 사용하였다. 기존의 분류기는 새로 초기화하였다.

## 5. 학습 전 평가

pipeline 추론 결과를 정답과 같은 형식의 레이블 목록으로 변환합니다. 예: 정답이 0/1이라면 예측도 0/1로 변환합니다. 생성형 모델이 유효하지 않은 답을 출력하면 정답과 일치하지 않는 레이블로 처리하고 평가에서 제외하지 않습니다.

아래 두 TODO를 완성하면 Accuracy와 결과 표는 자동 계산됩니다. Test 전체에서 평가하며, 정답을 추론 입력에 포함하지 않습니다. 학습 후에도 같은 데이터 순서·전처리·추론 설정을 사용합니다. 분류 외 태스크는 지표를 변경합니다.


In [7]:
def get_targets(test_data):
    """반환: Test 순서대로 정답 레이블을 담은 list."""
    return list(test_data["label"])

def predict_labels(current_model, processor, test_data):
    """반환: Test 순서대로 예측 레이블을 담은 list."""
    current_model.eval()

    classifier = pipeline(
        "text-classification",
        model=current_model,
        tokenizer=processor,
        device=0 if torch.cuda.is_available() else -1,
    )

    outputs = classifier(
        list(test_data["title"]),
        batch_size=32,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    return [label2id.get(result["label"], -1) for result in outputs]

def accuracy(targets, predictions):
    if len(targets) == 0 or len(targets) != len(predictions):
        raise ValueError("정답과 예측은 길이가 같고 비어 있지 않아야 합니다.")
    return sum(a == b for a, b in zip(targets, predictions)) / len(targets)

def evaluate_model(current_model, processor, test_data):
    targets = get_targets(test_data)
    predictions = predict_labels(current_model, processor, test_data)
    if len(targets) != len(test_data):
        raise ValueError("Test의 모든 샘플에 대해 정답과 예측을 반환하세요.")
    metrics = {"accuracy": accuracy(targets, predictions)}
    table = pd.DataFrame({"sample_id": range(len(targets)), "target": targets,
                          "prediction": predictions})
    return metrics, table

before_metrics, before = evaluate_model(model, processor, raw["test"])
print(before_metrics)
display(before.head())


{'accuracy': 0.1674693520140105}


,sample_id,target,prediction
0,0,3,5
1,1,1,6
2,2,5,0
3,3,0,4
4,4,5,0


## 6. LoRA 적용

모델의 모듈 이름을 확인한 뒤 target_modules, r, lora_alpha 등을 설정합니다. task_type은 해당 PEFT 지원 방식에 맞게 선택합니다. Classification head를 학습해야 한다면 modules_to_save 등 적절한 방법으로 포함하고, 저장 대상도 확인합니다.

In [8]:
from peft import LoraConfig, TaskType, get_peft_model


def make_lora_config(model):
    # 현재 모델에 적용할 모듈이 실제로 있는지 확인
    targets = [
        name for name, module in model.named_modules()
        if name.rsplit(".", 1)[-1] in {"query", "value"}
        and isinstance(module, torch.nn.Linear)
    ]
    assert targets, "LoRA를 적용할 query/value 모듈을 찾지 못했습니다."

    print("LoRA 대상 모듈 수:", len(targets))
    print("대상 예시:", targets[:4])

    return LoraConfig(
        task_type=TaskType.SEQ_CLS,
        target_modules=targets,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        modules_to_save=["classifier"],
    )


# 실수로 같은 모델에 두 번 적용하는 것을 방지
assert not hasattr(model, "peft_config"), "이미 LoRA가 적용된 모델입니다."

lora_config = make_lora_config(model)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# LoRA와 분류기만 학습 대상으로 열렸는지 확인
trainable_names = [
    name for name, param in model.named_parameters()
    if param.requires_grad
]

assert any("lora_" in name for name in trainable_names)
assert any("classifier" in name for name in trainable_names)
assert all(
    "lora_" in name or "classifier" in name
    for name in trainable_names
)

print("학습 대상 확인 완료: LoRA + 뉴스 분류기")

LoRA 대상 모듈 수: 24
대상 예시: ['bert.encoder.layer.0.attention.self.query', 'bert.encoder.layer.0.attention.self.value', 'bert.encoder.layer.1.attention.self.query', 'bert.encoder.layer.1.attention.self.value']
trainable params: 300,295 || all params: 110,923,022 || trainable%: 0.2707
학습 대상 확인 완료: LoRA + 뉴스 분류기


LoRA 설정 이유 및 실제 학습 대상 모듈: Query·Value 24개 모듈에 r=8, alpha=16, dropout=0.1을 적용했고, 학습 부담을 줄이면서 적응시키기 위한 시작 설정이다. 실제 학습 대상은 LoRA와 분류기 300,295개이다.

## 7. 학습

TrainingArguments, Data collator, Trainer를 구성합니다. Train은 학습에, Validation은 설정 선택에 사용합니다. Test는 전달하지 않습니다. Padding·labels 및 Loss가 태스크와 맞는지 확인합니다.

In [9]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)


def compute_metrics(result):
    predictions = result.predictions.argmax(axis=-1)
    return {
        "accuracy": float(
            (predictions == result.label_ids).mean()
        )
    }


def make_trainer(model, processor, train_data, validation_data):
    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=3e-4,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        fp16=True,
        label_names=["labels"],
        report_to="none",
        seed=SEED,
    )

    return Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=validation_data,
        processing_class=processor,
        data_collator=DataCollatorWithPadding(
            tokenizer=processor,
        ),
        compute_metrics=compute_metrics,
    )


trainer = make_trainer(
    model,
    processor,
    prepared["train"],
    prepared["validation"],
)

train_result = trainer.train()
print(train_result.metrics)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.723007,0.407305,0.854727
2,0.678640,0.383958,0.863402
3,0.644471,0.362875,0.870539


{'train_runtime': 448.1499, 'train_samples_per_second': 275.198, 'train_steps_per_second': 8.602, 'total_flos': 1324046242269720.0, 'train_loss': 0.7752399392628948, 'epoch': 3.0}


## 8. 학습 후 평가

학습 전 평가와 동일한 함수·Test 데이터를 사용합니다. 성능 향상이 없거나 실패한 사례도 그대로 기록합니다.

In [10]:
after_metrics, after = evaluate_model(model, processor, raw["test"])
assert len(before) >= 5 and len(after) >= 5, "비교 샘플을 최소 5개 기록하세요."
assert before["sample_id"].is_unique and after["sample_id"].is_unique
assert before[["sample_id", "target"]].reset_index(drop=True).equals(
    after[["sample_id", "target"]].reset_index(drop=True)
), "학습 전후 동일한 샘플·순서·정답을 사용하세요."
comparison = before.rename(columns={"prediction": "before"}).merge(
    after[["sample_id", "prediction"]].rename(columns={"prediction": "after"}),
    on="sample_id", validate="one_to_one"
)
display(pd.DataFrame({"before": before_metrics, "after": after_metrics}))
display(comparison)


,before,after
accuracy,0.167469,0.883319


,sample_id,target,before,after
0,0,3,5,3
1,1,1,6,1
2,2,5,0,5
3,3,0,4,0
4,4,5,0,5
...,...,...,...,...
4563,4563,1,6,4
4564,4564,2,4,2
4565,4565,4,4,4
4566,4566,4,4,4


## 9. 저장과 분석

Adapter를 저장합니다. 재사용에는 호환되는 Base model과 Processor·설정도 필요합니다. 추가 학습한 Classification head가 있다면 함께 저장되는지 확인합니다.

In [11]:
from pathlib import Path

adapter_dir = Path(OUTPUT_DIR) / "adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
model.config.save_pretrained(adapter_dir)
print("Base model:", MODEL_ID)
print("Adapter·Processor·모델 설정 저장 위치:", adapter_dir.resolve())


Base model: klue/bert-base
Adapter·Processor·모델 설정 저장 위치: /content/assignment_runs/news/adapter


In [12]:
sample_review = comparison.copy()
sample_review["title"] = list(raw["test"]["title"])

for column in ["target", "before", "after"]:
    sample_review[column] = sample_review[column].map(
        lambda x: id2label.get(x, "알 수 없는 출력")
    )

display(
    sample_review[
        ["sample_id", "title", "target", "before", "after"]
    ].head(10)
)

,sample_id,title,target,before,after
0,0,신동엽 50주기 맞아 산문 전집·앤솔로지 출간,생활문화,스포츠,생활문화
1,1,증시 또 패닉…코스피 2%·코스닥 3%대 급락종합,경제,정치,경제
2,2,프로농구 1위 DB·2위 KCC 나란히 승리…DB 매직넘버...,스포츠,IT과학,스포츠
3,3,구글 3세대 크롬캐스트 출시…신세계아이앤씨 단독 판매종합,IT과학,세계,IT과학
4,4,NBA 미네소타 덴버 꺾고 14년 만에 플레이오프행 감...,스포츠,IT과학,스포츠
5,5,최정 재역전포 SK 한화에 1경기 차 추격…두산은 5연승,스포츠,IT과학,스포츠
6,6,경찰 드루킹 출판사 무단침입 의혹 기자들 수사중,사회,스포츠,사회
7,7,與 인재공약 투트랙 발표…설 밥상민심 잡기 선제전략,사회,정치,정치
8,8,佛검찰 마크롱 전 보좌관 외교여권 부정사용 혐의로 체포,세계,스포츠,세계
9,9,더민주·국민의당 차기 광주시당 위원장 누가 맡나,정치,정치,정치


### 1. 실험 구성

한국어 뉴스 제목을 7개 주제로 분류하기 위해 `klue/bert-base`와 KLUE YNAT 데이터셋을 사용했다. Train 41,110개, Validation 9,107개, 실습용 Test 4,568개로 구성했으며, Test는 학습과 설정 선택에 사용하지 않았다.

BERT Attention의 Query·Value에 Rank 8 LoRA를 적용했다. 기존 BERT 가중치는 고정하고 LoRA와 새 분류기를 함께 학습했다. 학습 대상은 300,295개로 전체 파라미터의 약 0.2707%였다.

학습률은 3e-4, 학습 횟수는 3 Epoch로 설정했다. Batch size 16과 Gradient accumulation 2를 사용하여 GPU 한 개 기준 유효 Batch size는 32였다.

### 2. 학습 전후 성능

동일한 Test 데이터와 평가 함수를 사용한 결과, Accuracy는 16.75%에서 88.33%로 약 71.59%p 상승했다.

학습 전 분류기는 무작위로 초기화된 상태였다. 따라서 이 결과는 LoRA와 새 분류기를 함께 학습한 효과이며, 성능 향상을 LoRA만의 효과로 해석할 수는 없다. 이를 구분하려면 분류기만 학습하는 대조 실험이 필요하다.

### 3. 학습 과정

Validation Loss는 0.407305 → 0.383958 → 0.362875로 감소했고, Validation Accuracy는 85.47% → 86.34% → 87.05%로 상승했다.

관찰한 3 Epoch에서는 Validation Loss가 증가하는 뚜렷한 과적합 징후가 보이지 않았다. 다만 이 기록만으로 과적합이 없다고 단정할 수는 없다. 표의 Training Loss는 로깅 시점의 값으로, Epoch 전체 평균을 의미하지 않는다.

### 4. 한계와 개선 방향

단일 Seed와 한 번의 분할로 진행한 실험이므로 결과의 안정성을 충분히 확인하지 못했다. 또한 실습용으로 분리한 Test 결과이므로 KLUE 공식 Test 점수와 직접 비교할 수 없다.

후속 실험에서는 클래스별 성능과 오답을 분석하고, Validation을 기준으로 설정을 비교할 수 있다. LoRA와 분류기는 저장했지만, 저장한 Adapter를 다시 불러와 동일한 예측이 나오는지는 아직 확인하지 않았다.